### Transformer Theoretical Model

This notebook stores a bunch of analysis about a Transformer, e.g. estimates the number of FLOPs, parameters, peak memory footprint, checkpoint size, etc.

In [1]:
from collections import OrderedDict

In [2]:







block_size = 1024
vocab_size = 50257
n_layer = 12
n_head = 12
n_embd = 768
bias = False
assert not bias, "this notebook assumes bias=False just for simplicity"

In [3]:
def params():
    """ estimates the number of parameters in the model"""
    out = OrderedDict()

    
    out['emebedding/position'] = n_embd * block_size
    out['embedding/token'] = n_embd * vocab_size
    out['embedding'] = out['emebedding/position'] + out['embedding/token']

    
    out['attention/ln'] = n_embd 
    out['attention/kqv'] = n_embd * 3*n_embd
    out['attention/proj'] = n_embd**2
    out['attention'] = out['attention/ln'] + out['attention/kqv'] + out['attention/proj']

    
    ffw_size = 4*n_embd 
    out['mlp/ln'] = n_embd
    out['mlp/ffw'] = n_embd * ffw_size
    out['mlp/proj'] = ffw_size * n_embd
    out['mlp'] = out['mlp/ln'] + out['mlp/ffw'] + out['mlp/proj']
    
    
    out['block'] = out['attention'] + out['mlp']
    out['transformer'] = n_layer * out['block']
    out['ln_f'] = n_embd 
    out['dense'] = 0 

    
    out['total'] = out['embedding'] + out['transformer'] + out['ln_f'] + out['dense']

    return out


p = params()
params_total = p['total']
print(f"we see: {params_total}, expected: {124337664}, match: {params_total == 124337664}")

print(f"{'name':20s} {'params':10s} {'ratio (%)':10s}")
for k,v in p.items():
    print(f"{k:20s} {v:10d} {v/params_total*100:10.4f}")
    

we see: 124337664, expected: 124337664, match: True
name                 params     ratio (%) 
emebedding/position      786432     0.6325
embedding/token        38597376    31.0424
embedding              39383808    31.6749
attention/ln                768     0.0006
attention/kqv           1769472     1.4231
attention/proj           589824     0.4744
attention               2360064     1.8981
mlp/ln                      768     0.0006
mlp/ffw                 2359296     1.8975
mlp/proj                2359296     1.8975
mlp                     4719360     3.7956
block                   7079424     5.6937
transformer            84953088    68.3245
ln_f                        768     0.0006
dense                         0     0.0000
total                 124337664   100.0000


In [4]:


params_bytes = params_total*4
params_and_buffers_bytes = params_bytes + 2*params_bytes
print(f"est checkpoint size: {params_and_buffers_bytes/1e9:.2f} GB")
measured_bytes = 1542470366 
print(f"measured with wc -c ckpt.pt: {measured_bytes}")
print(f"fluff ratio: {measured_bytes/params_and_buffers_bytes*100:.2f}%")

est checkpoint size: 1.49 GB
measured with wc -c ckpt.pt: 1542470366
fluff ratio: 103.38%


We can also estimate the ratio of our GPU memory that will be taken up just by the weights and the buffers inside the AdamW optimizer

In [5]:
gpu_memory = 40e9 
print(f"memory ratio taken up just for parameters: {params_and_buffers_bytes / gpu_memory * 100:.2f}%")

memory ratio taken up just for parameters: 3.73%


i.e. not that much of the memory for this tiny model, most of the memory is activations (forward and backward). This of course changes dramatically for larger and larger models.

Let's estimate FLOPs for a single forward pass.

In [6]:
def flops():
    
    
    

    out = OrderedDict()
    head_size = n_embd // n_head

    
    
    out['attention/kqv'] = 2 * block_size * (n_embd * 3*n_embd)
    
    out['attention/scores'] = 2 * block_size * block_size * n_embd
    
    out['attention/reduce'] = 2 * n_head * (block_size * block_size * head_size)
    
    out['attention/proj'] = 2 * block_size * (n_embd * n_embd)
    out['attention'] = sum(out['attention/'+k] for k in ['kqv', 'scores', 'reduce', 'proj'])

    
    ffw_size = 4*n_embd 
    out['mlp/ffw1'] = 2 * block_size * (n_embd * ffw_size)
    out['mlp/ffw2'] = 2 * block_size * (ffw_size * n_embd)
    out['mlp'] = out['mlp/ffw1'] + out['mlp/ffw2']

    
    out['block'] = out['attention'] + out['mlp']
    out['transformer'] = n_layer * out['block']
    out['dense'] = 2 * block_size * (n_embd * vocab_size)

    
    out['forward_total'] = out['transformer'] + out['dense']
    out['backward_total'] = 2 * out['forward_total'] 
    out['total'] = out['forward_total'] + out['backward_total']

    return out
    

f = flops()
flops_total = f['forward_total']
print(f"{'name':20s} {'flops':14s} {'ratio (%)':10s}")
for k,v in f.items():
    print(f"{k:20s} {v:14d} {v/flops_total*100:10.4f}")
    

name                 flops          ratio (%) 
attention/kqv            3623878656     1.2426
attention/scores         1610612736     0.5522
attention/reduce         1610612736     0.5522
attention/proj           1207959552     0.4142
attention                8053063680     2.7612
mlp/ffw1                 4831838208     1.6567
mlp/ffw2                 4831838208     1.6567
mlp                      9663676416     3.3135
block                   17716740096     6.0747
transformer            212600881152    72.8963
dense                   79047426048    27.1037
forward_total          291648307200   100.0000
backward_total         583296614400   200.0000
total                  874944921600   300.0000


In [7]:


def palm_flops():
    """estimate of the model flops following PaLM paper formula"""
    
    
    N = params()['total'] - params()['emebedding/position']
    L, H, Q, T = n_layer, n_head, n_embd//n_head, block_size
    mf_per_token = 6*N + 12*L*H*Q*T
    mf = mf_per_token * block_size
    return mf

print(f"palm_flops: {palm_flops():d}, flops: {flops()['total']:d}, ratio: {palm_flops()/flops()['total']:.4f}")

palm_flops: 875062886400, flops: 874944921600, ratio: 1.0001


Ok they are quite similar, giving some confidence that my math in flops() function was ~ok. Now, A100 is cited at 312TFLOPS bfloat16 on tensor cores. So what is our model flops utilization (MFU)? I trained the model above with a batch_size of 20 and grad_accum of 5, which runs in about 755ms on a single A100 GPU. We get:

In [8]:

batch_size = 20 * 5 
measured_time = 0.755 
measured_throughput = batch_size / measured_time
flops_achieved = f['total'] * measured_throughput


a100_flops_promised = 312e12


print(f"fraction of A100 used: {flops_achieved / a100_flops_promised * 100:.2f}%")

fraction of A100 used: 37.14%


For reference, we'd prefer to be somewhere around 50%+, and not just for a single GPU but for an entire DDP run. So we still have some work to do, but at least we're within a factor of ~2X of what is achievable with this GPU.

In [9]:

model_size = params()['total'] 
tokens_num = 300e9 
a100_flops = 312e12 
assumed_mfu = 0.3 
flops_throughput = a100_flops * 8 * assumed_mfu 
flops_needed = 6 * model_size * tokens_num 
time_needed_s = flops_needed / flops_throughput 
print(f"time needed to train the model: {time_needed_s/3600/24:.2f} days")

time needed to train the model: 3.46 days


This is not a bad estimate at all. I trained this model and it converged in roughly 4 days. Btw as a good reference for where 6ND comes from and some intuition around it I recommend [Dzmitry's post](https://medium.com/@dzmitrybahdanau/the-flops-calculus-of-language-model-training-3b19c1f025e4).

Now, FLOPs are just one constraint, the other that we have to keep a close track of is the memory bandwidth. TODO estimate LOAD/STORE costs of our model later.